In [1]:
# импортируем библиотеки, которые пригодятся для задачи
import torch
import torch.nn as nn
import re
import random
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import BertTokenizerFast
from tqdm import tqdm
from sklearn.model_selection import train_test_split


random.seed(42)
torch.manual_seed(42)

/Users/emc2/Проекты/личное/Практикум/Deep learning Engineer/practicum_dle/Sprint 2/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [105]:
# функция для "чистки" текстов
def clean_string(text):
    # приведение к нижнему регистру
    text = text.lower()
    # удаление всего, кроме латинских букв, цифр и пробелов
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # удаление дублирующихся пробелов, удаление пробелов по краям
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# загружаем датасет WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# длины последовательностей в датасете
# seq_len = 7 => 3 токена до <MASK> + токен <MASK> + 3 токена после
seq_len = 7

# удаляем слишком короткие тексты
texts = [line for line in dataset["text"] if len(line.split()) >= seq_len]

# "чистим" тексты
cleaned_texts = list(map(clean_string, texts))

# для упрощения используем только max_texts_count текстов
max_texts_count = 7000

# разбиение на тренировочную и валидационную выборки
val_size = 0.05

train_texts, val_texts = train_test_split(cleaned_texts[:max_texts_count], test_size=val_size, random_state=42)
print(f"Train texts: {len(train_texts)}, Val texts: {len(val_texts)}")


# класс датасета
class MaskedBertDataset(Dataset):
    def __init__(self, texts, tokenizer, seq_len=7):
        # self.samples - список пар (x, y)
        # x - токенизированный текст с пропущенным токеном
        # y - пропущенный токен
        self.samples = []

        for line in texts:
            token_ids = tokenizer.encode(line, add_special_tokens=False, max_length=512, truncation=True) # токенизируйте строку line

            # если строка слишком короткая, то пропускаем её
            if len(token_ids) < seq_len:
                continue

            # проходимся по всем токенам в последовательности
            for i in range(1, len(token_ids) - 1):
                '''
                context - список из seq_len // 2 токенов до i-го токена, токена tokenizer.mask_token_id, и seq_len // 2 токенов после i-го токена
                '''
                start_idx = max(0, i - seq_len // 2)
                end_idx = i + 1 + seq_len // 2
                context = (
                    token_ids[start_idx:i]
                    + [tokenizer.mask_token_id]
                    + token_ids[i+1:end_idx]
                )  # соберите контекст вокруг i-го токена

                # если контекст слишком короткий, то пропускаем его
                if len(context) < seq_len:
                    continue
                target = token_ids[i]  # возьмите i-ый токен последовательности
                self.samples.append((context, target))
           
    def __len__(self):
        return len(self.samples)  # верните размер датасета

    def __getitem__(self, idx):
        x, y = self.samples[idx]  # получите контекст и таргет для элемента с индексом idx
        return torch.tensor(x), torch.tensor(y)

# загружаем токенизатор
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# тренировочный и валидационный датасеты
train_dataset = MaskedBertDataset(train_texts, tokenizer, seq_len=seq_len)
val_dataset = MaskedBertDataset(val_texts, tokenizer, seq_len=seq_len)

# даталоадеры
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

print(f"Train dataset size: {len(train_dataset)}, Val dataset size: {len(val_dataset)}")
print(f"Train loader size: {len(train_loader)}, Val loader size: {len(val_loader)}")


Train texts: 6650, Val texts: 350
Train dataset size: 592449, Val dataset size: 29617
Train loader size: 9258, Val loader size: 463


In [106]:
class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, rnn_type="GRU", combine="concat"):
        super().__init__()
        # сохраняем значение combine
        self.combine = combine
        # входная размерность эмбеддинг-слоя - vocab_size, выходная - hidden_dim
        self.embedding = nn.Embedding(vocab_size, hidden_dim)  # ембеддинг слой
        # здесь должны быть разные блоки в зависимости от rnn_type
        # двунаправленный рекуррентный блок
        rnn_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type]
        self.rnn = rnn_cls(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)


        # out_dim может быть разным в зависимости от значения combine
        out_dim = 2*hidden_dim if combine == 'concat' else hidden_dim  # размер выходного вектора после агрегации скрытых состояний

        # выходной линейный слой
        self.fc = nn.Linear(out_dim, vocab_size)


    def forward(self, x):
        emb = self.embedding(x)  # результат эмбеддинг-слоя
        out, _ = self.rnn(emb)  # результат рекуррентного слоя

        center = x.size(1) // 2 # позиция центрального <MASK> токена

        # скрытые состояния <MASK> токена 
        # после двух проходов двунаправленяной сети
        hidden_forward = out[:, center, :out.size(2)//2]
        hidden_backward = out[:, center, out.size(2)//2:]

        # агрегация скрытых состояний в зависимости от self.combine
        # конкатенация или сумма hidden_forward и hidden_backward
        hidden_agg = hidden_forward + hidden_backward
        if self.combine == 'concat':
            hidden_agg = torch.cat([hidden_forward, hidden_backward], dim=1)

        linear_out = self.fc(hidden_agg)
        return linear_out


# Подсчёт параметров
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# размер словаря
vocab_size = tokenizer.vocab_size  

# размер скрытого состояния
hidden_dim = 128

# типы рекуррентных блоков
rnn_types = ["RNN", "GRU", "LSTM"]

# методы агрегаций скрытых состояний
combine_methods = ["sum", "concat"]

# Сравнение
print(f"{'RNN Type':<8} | {'Combine':<8} | {'Params':>10}")
print("-" * 35)
for rnn_type in rnn_types:
    for combine in combine_methods:
        model = BiRNNClassifier(vocab_size, hidden_dim, rnn_type, combine)  # объект модели с блоком rnn_type и агрегацией combine
        param_count = count_parameters(model)  # количество параметров модели
        print(f"{rnn_type:<8} | {combine:<8} | {param_count:>10,}")

RNN Type | Combine  |     Params
-----------------------------------
RNN      | sum      |  7,910,202
RNN      | concat   | 11,817,018
GRU      | sum      |  8,042,298
GRU      | concat   | 11,949,114
LSTM     | sum      |  8,108,346
LSTM     | concat   | 12,015,162


In [107]:
# объекты модели, оптимизатора, функции потерь
model = BiRNNClassifier(vocab_size, rnn_type="LSTM", combine="concat")
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()

# функция замера лосса и accuracy
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    sum_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_output = model(x_batch)  # выход модели для входа x_batch
            loss = criterion(x_output, y_batch)  # функция потерь
            preds = torch.argmax(x_output, dim=1)  # предсказанные токены
            correct += (preds == y_batch).sum().item()  # количество верно угаданных токенов
            total += y_batch.size(0)  # размер батча
            sum_loss += loss  # суммарная функция потерь
    
    # лосс и accuracy
    avg_loss = sum_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


# Основной цикл обучения
n_epochs = 3

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.
    for x_batch, y_batch in tqdm(train_loader):
        optimizer.zero_grad()  # обнуление градиентов оптимизатора
        x_output = model(x_batch)  # выход модели для входа x_batch
        loss = criterion(x_output, y_batch)  # функция потерь
        loss.backward()  # расчёт градиентов
        optimizer.step()  # обновление градиентов
        train_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f} | Val Accuracy: {val_acc:.2%}")

100%|██████████| 9258/9258 [06:31<00:00, 23.67it/s]


Epoch 1 | Train Loss: 6.289 | Val Loss: 5.709 | Val Accuracy: 22.12%


100%|██████████| 9258/9258 [06:35<00:00, 23.43it/s]


Epoch 2 | Train Loss: 4.885 | Val Loss: 5.519 | Val Accuracy: 24.57%


100%|██████████| 9258/9258 [06:32<00:00, 23.62it/s]


Epoch 3 | Train Loss: 4.118 | Val Loss: 5.555 | Val Accuracy: 25.47%


In [108]:
model.eval()
bad_cases, good_cases = [], []
with torch.no_grad():
    for x_batch, y_batch in val_loader:
        logits = model(x_batch)
        preds = torch.argmax(logits, dim=1)
        for i in range(len(y_batch)):
            input_tokens = tokenizer.convert_ids_to_tokens(x_batch[i].tolist())
            true_tok = tokenizer.convert_ids_to_tokens([y_batch[i].item()])[0]
            pred_tok = tokenizer.convert_ids_to_tokens([preds[i].item()])[0]
            
            if preds[i] != y_batch[i]:
                bad_cases.append((input_tokens, true_tok, pred_tok))
            else:
                good_cases.append((input_tokens, true_tok, pred_tok))


random.seed(42)
bad_cases_sampled = random.sample(bad_cases, 5)
good_cases_sampled = random.sample(good_cases, 5)


print("\nSome incorrect predictions:")
for context, true_tok, pred_tok in bad_cases_sampled:
    print(f"Input: {' '.join(context)} | True: {true_tok} | Predicted: {pred_tok}")


print("\nSome correct predictions:")
for context, true_tok, pred_tok in good_cases_sampled:
    if true_tok == pred_tok:
        print(f"Input: {' '.join(context)} | True: {true_tok} | Predicted: {pred_tok}")


Some incorrect predictions:
Input: regained independence in [MASK] whose adolescence was | True: 1918 | Predicted: 1913
Input: can always be [MASK] into some sort | True: transformed | Predicted: divided
Input: to change its [MASK] but were unable | True: name | Predicted: range
Input: uk schools has [MASK] largely at the | True: declined | Predicted: been
Input: the species as [MASK] in broom ##sed | True: occurring | Predicted: well

Some correct predictions:
Input: near ark ##ade [MASK] ##hia on the | True: ##lp | Predicted: ##lp
Input: under the control [MASK] the soviet union | True: of | Predicted: of
Input: ##rov would be [MASK] 13th addition to | True: the | Predicted: the
Input: hatch ##ery in [MASK] town of el | True: the | Predicted: the
Input: believes there to [MASK] cheese down there | True: be | Predicted: be
